## FINETUNE V1.1 FOR GENERAL DATA

Attach one Sentence Transformers v1 model and one SQLite database containing the `general_triplet` table to this Kaggle notebook, then run all cells. The final model is written to `/kaggle/working/vietnamese-embedding-v1.1-general`.

In [ ]:
import os
import sqlite3
from pathlib import Path

import sentence_transformers
import torch

KAGGLE_INPUT_DIR = Path('/kaggle/input')
KAGGLE_WORKING_DIR = Path('/kaggle/working')

# Normally these can stay as None. Set an absolute path only when more than
# one compatible model or database is attached to the notebook.
MODEL_DIR_OVERRIDE = None
DB_PATH_OVERRIDE = None

CHECKPOINT_DIR = KAGGLE_WORKING_DIR / 'vietnamese-embedding-v1.1-general-checkpoints'
FINAL_MODEL_DIR = KAGGLE_WORKING_DIR / 'vietnamese-embedding-v1.1-general'


def has_table(database_path: Path, table_name: str) -> bool:
    try:
        uri = f'file:{database_path.resolve()}?mode=ro'
        with sqlite3.connect(uri, uri=True) as connection:
            row = connection.execute(
                'SELECT 1 FROM sqlite_master WHERE type = ? AND name = ?',
                ('table', table_name),
            ).fetchone()
        return row is not None
    except sqlite3.Error:
        return False


def select_unique_path(candidates: list[Path], kind: str) -> Path:
    if not candidates:
        raise FileNotFoundError(f'No compatible {kind} found under {KAGGLE_INPUT_DIR}')
    if len(candidates) > 1:
        choices = '\n'.join(f'  - {path}' for path in candidates)
        raise RuntimeError(
            f'Multiple compatible {kind} paths were found. Set the corresponding '
            f'override at the top of this cell:\n{choices}'
        )
    return candidates[0]


if not KAGGLE_INPUT_DIR.is_dir() or not KAGGLE_WORKING_DIR.is_dir():
    raise RuntimeError('This notebook is configured to run on Kaggle.')

if MODEL_DIR_OVERRIDE is None:
    model_candidates = sorted({path.parent for path in KAGGLE_INPUT_DIR.rglob('modules.json')})
    MODEL_DIR = select_unique_path(model_candidates, 'Sentence Transformers model')
else:
    MODEL_DIR = Path(MODEL_DIR_OVERRIDE)
    if not (MODEL_DIR / 'modules.json').is_file():
        raise FileNotFoundError(f'Invalid Sentence Transformers model directory: {MODEL_DIR}')

if DB_PATH_OVERRIDE is None:
    database_candidates = sorted(
        path
        for path in KAGGLE_INPUT_DIR.rglob('*')
        if path.is_file() and path.suffix.lower() in {'.db', '.sqlite', '.sqlite3'}
        and has_table(path, 'general_triplet')
    )
    DB_PATH = select_unique_path(database_candidates, 'general_triplet database')
else:
    DB_PATH = Path(DB_PATH_OVERRIDE)
    if not DB_PATH.is_file() or not has_table(DB_PATH, 'general_triplet'):
        raise FileNotFoundError(f'Invalid general_triplet database: {DB_PATH}')

world_size = int(os.environ.get('WORLD_SIZE', '1'))
if world_size != 1:
    raise RuntimeError(
        'Run this notebook normally with Run All; do not launch it with torchrun. '
        'The single notebook process will use all visible Kaggle GPUs.'
    )

gpu_count = torch.cuda.device_count()
if gpu_count == 0:
    raise RuntimeError('No CUDA GPU detected. Enable a GPU accelerator in Kaggle.')

print(f'Sentence Transformers: {sentence_transformers.__version__}')
print(f'Model input: {MODEL_DIR}')
print(f'Database input: {DB_PATH}')
print(f'Final model output: {FINAL_MODEL_DIR}')
for index in range(gpu_count):
    properties = torch.cuda.get_device_properties(index)
    memory_gib = properties.total_memory / 1024**3
    print(f'cuda:{index}: {properties.name} ({memory_gib:.1f} GiB)')

In [ ]:
from datasets import Features, IterableDataset, Value

ELIGIBLE_ROWS = """
    FROM general_triplet
    WHERE anchor IS NOT NULL
      AND positive IS NOT NULL
      AND hard_negative IS NOT NULL
"""


def open_database() -> sqlite3.Connection:
    uri = f'file:{DB_PATH.resolve()}?mode=ro'
    connection = sqlite3.connect(uri, uri=True)
    connection.row_factory = sqlite3.Row
    return connection


def get_num_records() -> int:
    with open_database() as connection:
        return connection.execute(
            f'SELECT COUNT(*) {ELIGIBLE_ROWS}'
        ).fetchone()[0]


def get_data(fetch_size: int = 10_000):
    connection = open_database()
    cursor = connection.execute(
        f"""
        SELECT anchor, positive, hard_negative
        {ELIGIBLE_ROWS}
        ORDER BY RANDOM()
        """
    )

    try:
        while rows := cursor.fetchmany(fetch_size):
            for row in rows:
                yield {
                    'anchor': f"query: {row['anchor']}",
                    'positive': f"passage: {row['positive']}",
                    'hard_negative': f"passage: {row['hard_negative']}",
                }
    finally:
        cursor.close()
        connection.close()


train_features = Features({
    'anchor': Value('string'),
    'positive': Value('string'),
    'hard_negative': Value('string'),
})
train_dataset = IterableDataset.from_generator(
    get_data,
    features=train_features,
)

In [ ]:
import math

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,
)

BATCH_SIZE_PER_DEVICE = 32
EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 1

num_records = get_num_records()
if num_records == 0:
    raise RuntimeError('The database contains no complete training triplets.')

samples_per_micro_batch = BATCH_SIZE_PER_DEVICE * gpu_count
micro_batches_per_epoch = math.ceil(num_records / samples_per_micro_batch)
optimizer_steps_per_epoch = math.ceil(
    micro_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
)
max_steps = optimizer_steps_per_epoch * EPOCHS
effective_batch_size = samples_per_micro_batch * GRADIENT_ACCUMULATION_STEPS

print(f'Records per epoch: {num_records:,}')
print(f'Complete data passes: {EPOCHS}')
print(f'Expected row visits: {num_records * EPOCHS:,}')
print(f'Batch size per GPU: {BATCH_SIZE_PER_DEVICE}')
print(f'Effective batch size: {effective_batch_size}')
print(f'Optimizer steps per epoch: {optimizer_steps_per_epoch:,}')
print(f'Max optimizer steps: {max_steps:,}')

model = SentenceTransformer(str(MODEL_DIR))
loss = MultipleNegativesRankingLoss(model)

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    max_steps=max_steps,
    per_device_train_batch_size=BATCH_SIZE_PER_DEVICE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    dataloader_num_workers=0,
    dataloader_drop_last=False,
    dataloader_pin_memory=True,
    logging_steps=100,
    save_strategy='steps',
    save_steps=5_000,
    save_total_limit=2,
    report_to='none',
    seed=42,
    data_seed=42,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
)

In [ ]:
trainer.train()
trainer.save_model(str(FINAL_MODEL_DIR))

if not (FINAL_MODEL_DIR / 'modules.json').is_file():
    raise RuntimeError(f'Final model export failed: {FINAL_MODEL_DIR}')

print(f'Final model exported to: {FINAL_MODEL_DIR}')